# Систематизация Python: функции + ООП на примере шахмат


## 1. Начинаем без ООП: обычная функция

Сначала решим очень маленькую задачу.

Пусть координата клетки задается двумя числами от `0` до `7`.
Нужно проверить, существует ли такая клетка.


In [1]:
def is_inside(row: int, col: int) -> bool:
    """Проверить, находится ли клетка (row, col) внутри шахматной доски."""
    return 0 <= row <= 7 and 0 <= col <= 7


## 2. Функция для шахматной ладьи

Пока фигуры как объекта не существует.
Мы просто передаем координаты.


In [6]:
def rook_can_move(
    row: int,
    col: int,
    new_row: int,
    new_col: int
    ) -> bool:
    """
    Проверить, может ли ладья перейти из клетки (row, col)
    в клетку (new_row, new_col).

    Метод должен учитывать:
    - границы доски;
    - запрет хода в ту же клетку;
    - движение только по строке или столбцу.
    """
    can_move_1 = is_inside(row, col) and is_inside(new_row, new_col)
    can_move_2 = ((row == new_row and col != new_col) or (row != new_row and col == new_col))
    return can_move_1 and can_move_2



False


# 3. Первый класс: шахматная фигура

Объект удобен, когда **данные и действия над этими данными логически принадлежат одной сущности**.


In [7]:
class Piece:
    def __init__(self, color: str, row: int, col: int):
        """Сохранить цвет фигуры и её начальную позицию."""
        self.color = color
        self.row = row
        self.col = col

    def position(self) -> tuple[int, int]:
        """Вернуть текущие координаты фигуры."""
        return self.row, self.col

    def move_to(self, row: int, col: int) -> None:
        """Изменить позицию фигуры на указанную клетку. Без проверки на возможность движения."""
        self.row = row
        self.col = col


### Здесь необходимо остановиться и систематизировать

**Класс** — описание того, какими данными и поведением обладают объекты.

**Объект** — конкретный экземпляр класса.

**`__init__`** — метод, который задает начальное состояние объекта.

**`self`** — ссылка на конкретный объект, с которым сейчас работает метод.

Сравните:

```python
move_to(piece, 2, 0)
```

и

```python
piece.move_to(2, 0)
```

Второй вариант подчеркивает: действие принадлежит самой фигуре.


# 4. Магический метод `__str__`

Пусть фигура умеет удобно выводиться на экран.


In [19]:
class Piece:
    def __init__(self, color: str, row: int, col: int):
        """Сохранить цвет фигуры и её начальную позицию."""
        self.color = color
        self.row = row
        self.col = col

    def can_move(self, new_row: int, new_col: int) -> bool:
        pass

    def position(self) -> tuple[int, int]:
        """Вернуть текущие координаты фигуры."""
        return self.row, self.col

    def move_to(self, row: int, col: int) -> bool:
        """Изменить позицию фигуры на указанную клетку. Без проверки на возможность движения."""
        if not self.can_move(row, col):
            return False
        self.row = row
        self.col = col
        return True

    def __str__(self) -> str:
        """Вернуть строковое обозначение фигуры для вывода на доске."""
        return f"Фигура {self.__class__.__name__} {self.color} цвета на позиции {self.row}, {self.col}."

    def char(self):
        pass

p = Piece("WHITE", 1, 7)
print(p)

Фигура Piece WHITE цвета на позиции 1, 7.


`__str__` — пример специального метода.

Мы не вызываем его напрямую:

```python
piece.__str__()
```

Обычно мы пишем:

```python
print(piece)
```

а Python сам вызывает нужный метод.


# 5. Наследование: ладья

У всех шахматных фигур есть общие свойства.
Но правила движения отличаются.

Поэтому общую часть оставим в `Piece`, а конкретные фигуры будем наследовать.


In [24]:
class Rook(Piece):
    def can_move(self,
        new_row: int,
        new_col: int
        ) -> bool:
        """
        Проверить, может ли ладья перейти из клетки (row, col)
        в клетку (new_row, new_col).
    
        Метод должен учитывать:
        - границы доски;
        - запрет хода в ту же клетку;
        - движение только по строке или столбцу.
        """
        can_move_1 = is_inside(self.row, self.col) and is_inside(new_row, new_col)
        can_move_2 = ((self.row == new_row and self.col != new_col) or (self.row != new_row and self.col == new_col))
        return can_move_1 and can_move_2

    def char(self):
        return "R"

r1 = Rook("WHITE", 1, 1)
print(r1.move_to(5, 9))
print(r1)

False
Фигура Rook WHITE цвета на позиции 1, 1.


### Что дала нам наследуемость

`Rook` автоматически получил:

- `__init__`;
- `position`;
- `move_to`.

Но добавил собственный метод `can_move` и переопределил `__str__`.

Это и есть ключевая идея:

> общее поведение — в базовом классе, различия — в дочерних.


# 6. Добавляем коня


In [25]:
class Knight(Piece):
    def can_move(self, row: int, col: int) -> bool:
        """
        Проверить возможность хода коня.

        Конь должен менять одну координату на 1,
        а другую — на 2.
        """
        can_move_1 = (abs(self.row - row) == 2 and abs(self.col - col) == 1) or (abs(self.row - row) == 1 and abs(self.col - col) == 2)
        can_move_2 = is_inside(self.row, self.col) and is_inside(row, col)
        return can_move_1 and can_move_2

    def char(self):
        return "K"
        



# 7. Полиморфизм

Теперь самое важное.

У объектов разные классы, но мы можем обращаться к ним одинаково.


In [ ]:
# Создать список фигур разных классов.
# Для каждой фигуры вызвать одинаковые методы:
# - position()
# - can_move(...)



Мы **не спрашиваем**, является ли объект ладьей или конем.

Мы просто вызываем:

```python
piece.can_move(...)
```

Каждый объект сам знает, как выполнить этот метод.

Это хороший практический смысл полиморфизма.


# 8. Функция как объект

Повторим тему функций.

Функцию можно сохранить в переменную и передать другой функции.


In [23]:
def is_white(piece):
    """Вернуть True, если фигура белая."""
    return piece.color == "WHITE"


# Сохранить функцию в переменную и вызвать её через эту переменную.


# 9. Функция высшего порядка

Напишем функцию, которая получает:

1. список фигур;
2. другую функцию — условие отбора.


In [ ]:
def select_pieces(pieces, predicate) -> list[Piece]:
    """
    Вернуть только те фигуры, для которых predicate(piece) возвращает True.

    predicate — функция, переданная как аргумент.
    """
    pass


Тот же вызов можно выполнить через `lambda`:


In [ ]:
# Использовать select_pieces вместе с lambda,
# чтобы выбрать только чёрные фигуры.


Здесь соединяются две большие темы курса:

**ООП:** элементы списка — объекты.

**Функции:** условие отбора передается как отдельный объект-функция.


# 10. Класс шахматной доски

Теперь появляется новая сущность — доска.

Именно доска должна знать:

- какие фигуры на ней находятся;
- свободна ли клетка;
- какая фигура стоит на координате;
- как показать позицию.


In [34]:
class Board:
    def __init__(self):
        """Создать пустую шахматную доску и хранилище фигур."""
        self.table = [[None] * 8 for i in range(8)]

    def add_piece(self, piece):
        """Добавить фигуру на доску."""
        self.table[piece.row][piece.col] = piece

    def get_piece(self, row: int, col: int):
        """Вернуть фигуру на указанной клетке или None, если клетка пуста."""
        return self.table[row][col]

    def is_empty(self, row: int, col: int) -> bool:
        """Проверить, свободна ли указанная клетка."""
        return self.table[row][col] != None

    def show(self) -> None:
        """
        Вывести доску 8×8.

        Для пустой клетки вывести точку,
        для занятой — строковое обозначение фигуры.
        """
        print("", end="\t")
        for col in range(8):
            print(col, end="\t")
        print()
        print()
        print()
        for row in range(8):
            print(row, end="\t")
            for col in range(8):
                if self.table[row][col] == None:
                    print(".", end="\t")
                else:
                    if self.table[row][col].color == "WHITE":
                        print("w" + self.table[row][col].char(), end="\t")
                    elif self.table[row][col].color == "BLACK":
                        print("b" + self.table[row][col].char(), end="\t")
            print()
            print()
            print()


board = Board()
board.add_piece(Knight("WHITE", 1, 7))
board.add_piece(Rook("BLACK", 2, 3))
board.show()

	0	1	2	3	4	5	6	7	


0	.	.	.	.	.	.	.	.	


1	.	.	.	.	.	.	.	wK	


2	.	.	.	bR	.	.	.	.	


3	.	.	.	.	.	.	.	.	


4	.	.	.	.	.	.	.	.	


5	.	.	.	.	.	.	.	.	


6	.	.	.	.	.	.	.	.	


7	.	.	.	.	.	.	.	.	




# 11. Функция, которая работает с объектом

Не всё обязано быть методом класса.

Например, поиск всех геометрически допустимых ходов удобно вынести в обычную функцию.


In [ ]:
def possible_moves(piece) -> list[tuple[int, int]]:
    """
    Найти все клетки доски, в которые фигура
    геометрически может сделать ход.

    Состояние других фигур на доске пока не учитывать.
    """
    lst = []
    for row in range(8):
        for col in range(8):
            if piece.can_move(row, col):
                lst.append((row, col))
    return lst

# 12. Сортировка объектов через `key`

Еще раз связываем функции и ООП.


In [ ]:
# Создать несколько фигур и отсортировать их через sorted(..., key=...).
#
# key должен быть функцией, которая для каждой фигуры
# возвращает значение, по которому выполняется сортировка.


## Задание 1. Слон

Напишите класс `Bishop`.

Слон может ходить по диагонали.

Подсказка:

```python
abs(new_row - row) == abs(new_col - col)
```


In [35]:
class Bishop(Piece):
    def can_move(self, row: int, col: int) -> bool:
        can_move_1 = abs(row - self.row) == abs(col - self.col)
        can_move_2 = is_inside(self.row, self.col) and is_inside(row, col)
        can_move_3 = not(row == self.row and col == self.col)
        return can_move_1 and can_move_2 and can_move_3

    def char(self):
        return "B"


### Проверка


In [37]:
bishop = Bishop("WHITE", 2, 2)

# Должно быть:
# True
# True
# False

print(bishop.can_move(5, 5))
print(bishop.can_move(0, 4))
print(bishop.can_move(2, 5))


True
True
False


## Задание 2. Ферзь

Не пишите правило ферзя с нуля.

Ферзь объединяет правила:

- ладьи;
- слона.

Сначала проговорите, как это выразить логическим выражением.


In [38]:
class Queen(Piece):
    def can_move(self, row: int, col: int) -> bool:
        can_move_bishop_1 = abs(row - self.row) == abs(col - self.col)
        can_move_bishop_2 = not(row == self.row and col == self.col)
        can_move_rook_1 = ((self.row == row and self.col != col) or (self.row != row and self.col == col))
        can_move_1 = is_inside(self.row, self.col) and is_inside(row, col)
        return can_move_1 and (can_move_rook_1 or can_move_bishop_1 and can_move_bishop_2)

    def char(self):
        return "Q"


## Задание 3. Пешка

Для упрощенной модели:

- белая пешка идет на одну строку вверх;
- черная — на одну строку вниз;
- взятия пока не рассматриваем.

Здесь важно увидеть, что поведение зависит от **состояния объекта** — его цвета.


In [39]:
class Pawn(Piece):
    def can_move(self, row: int, col: int) -> bool:
        if not(is_inside(self.row, self.col) and is_inside(row, col)):
            return False
            
        if self.color == "WHITE":
            if self.row == 1:
                return ((self.row + 1 == row) or (self.row + 2 == row)) and (self.col == col)
            else:
                return (self.row + 1 == row) and (self.col == col)
        elif self.color == "BLACK":
            if self.row == 6:
                return ((self.row - 1 == row) or (self.row - 2 == row)) and (self.col == col)
            else:
                return (self.row - 1 == row) and (self.col == col)

    def char(self):
        return "P"


## Задание 4. Проверка свободного пути

Ладья, слон и ферзь не могут перепрыгивать через фигуры.

Напишем **обычную функцию**:

```python
is_path_clear(board, piece, row, col)
```

Она должна проверить промежуточные клетки.

Это специально делается функцией, чтобы еще раз разделить:

- правило движения фигуры;
- состояние доски.


In [ ]:
def is_path_clear(
    board: Board,
    piece: Piece,
    row: int,
    col: int
) -> bool:
    # TODO
    pass


## Задание 5. Перемещение фигуры

Добавьте в `Board` метод:

```python
move_piece(row, col, new_row, new_col)
```

Он должен:

1. найти фигуру;
2. проверить существование конечной клетки;
3. проверить `can_move`;
4. убедиться, что конечная клетка не занята своей фигурой;
5. для дальнобойных фигур проверить путь;
6. передвинуть фигуру.

На этом задании особенно хорошо повторяются:

- методы;
- `self`;
- `return`;
- взаимодействие объектов разных классов.


In [ ]:
# Допишите этот метод в Board.

# def move_piece(
#     self,
#     row: int,
#     col: int,
#     new_row: int,
#     new_col: int
# ) -> bool:
#     ...


## Задание 6. Все доступные ходы

Перепишите `possible_moves` так, чтобы функция учитывала доску.

Сигнатура:

```python
possible_moves(board, piece)
```

Результат — список клеток, куда фигура действительно может сделать ход.


In [ ]:
def possible_moves_on_board(
    board: Board,
    piece: Piece
) -> list[tuple[int, int]]:
    # TODO
    pass


## Задание 7. Функции-предикаты

Используя уже написанную `select_pieces`, получите:

1. все белые фигуры;
2. все черные фигуры;
3. все фигуры на нижней половине доски;
4. все ладьи.

Попробуйте часть условий записать обычными функциями, а часть — через `lambda`.


In [ ]:
# Пример начала:

white = select_pieces(
    board.pieces,
    lambda piece: piece.color == "white"
)

# TODO: еще три выборки.


## Задание 8. Оценка позиции

Добавим стоимость фигур:

- пешка — 1;
- конь — 3;
- слон — 3;
- ладья — 5;
- ферзь — 9.

Напишите функцию:

```python
piece_value(piece)
```

а затем:

```python
position_score(board)
```

Положительный результат означает преимущество белых,
отрицательный — преимущество черных.

Это хороший пример задачи, которую естественно решить **обычными функциями**, а не новым большим классом.


In [ ]:
def piece_value(piece: Piece) -> int:
    # TODO
    pass


def position_score(board: Board) -> int:
    # TODO
    pass


# Финальная сборка

В конце семинара создайте позицию минимум из 8 фигур:

```python
board = Board()
```

Затем:

1. выведите доску;
2. сделайте несколько допустимых ходов;
3. попробуйте недопустимый ход;
4. получите все ходы одной фигуры;
5. отберите белые фигуры;
6. посчитайте оценку позиции.
